# 🛡️ MetaGuard — GRPO Training Notebook

**Team:** Parth Singhal, Mehakveer Kaur, Kartik Goyal  
**HF Space:** https://huggingface.co/spaces/parth-1/MetaGuard  
**Hackathon:** OpenEnv — Meta × Scaler  

This notebook trains **Llama 3.1 8B** using GRPO on the MetaGuard Ad Policy Compliance environment.

### What this trains:
- Agent learns to follow structured SOP: `query_regulations → gather signals → submit_audit → decide`
- Reward shaped by correctness, sequence compliance, API failure recovery
- Environment runs locally in the notebook (fast); GPU handles only the model

## Cell 1 — Install Dependencies

In [ ]:
!pip install unsloth trl transformers datasets accelerate peft -q
!pip install openenv-core==0.2.1 --no-deps -q
!pip install fastapi uvicorn pydantic requests openai matplotlib -q
print('✅ Dependencies installed')

## Cell 2 — Clone Repo

In [ ]:
import os

if not os.path.exists('meta-ad-policy-sandbox'):
    !git clone https://github.com/Parth380/meta-ad-policy-sandbox.git

%cd meta-ad-policy-sandbox
!pip install -e . -q
os.makedirs('outputs', exist_ok=True)
print('Repo installed & outputs/ ready')

## Cell 3 — Config (SET THESE)

In [ ]:
import os

os.environ['ENV_URL']  = 'http://localhost:8000'  # local env (fast); change to HF Space URL if needed
os.environ['HF_REPO']  = 'parth-1/metaguard-llama3.1-8b-grpo'
os.environ['HF_TOKEN'] = ''  # paste your HF write token here

ENV_URL   = os.environ['ENV_URL']
HF_TOKEN  = os.environ['HF_TOKEN']
HF_REPO   = os.environ['HF_REPO']

print(f'ENV_URL  : {ENV_URL}')
print(f'HF_REPO  : {HF_REPO}')
print(f'HF_TOKEN : {"set" if HF_TOKEN else "MISSING -- set above before Cell 11"}')

## Cell 4 — Boot Local Environment

In [ ]:
import subprocess, time, threading, requests
import uvicorn

procs = [
    subprocess.Popen(['python', 'apps/regulatory_api.py']),
    subprocess.Popen(['python', 'apps/crm_api.py']),
    subprocess.Popen(['python', 'apps/audit_api.py']),
]
time.sleep(3)

from server.app import app as _env_app
threading.Thread(
    target=uvicorn.run,
    kwargs={'app': _env_app, 'host': '0.0.0.0', 'port': 8000, 'log_level': 'warning'},
    daemon=True,
).start()
time.sleep(2)

for i in range(20):
    try:
        r = requests.post(f'{ENV_URL}/reset', json={'task_id': 'task_1_healthcare'}, timeout=5)
        if r.status_code == 200:
            print(f'Environment ready (attempt {i+1})')
            break
    except:
        pass
    time.sleep(1)
else:
    raise RuntimeError('ENV not reachable after 20 attempts')

## Cell 5 — Imports + Helpers

In [ ]:
import json
import random
import torch
import matplotlib.pyplot as plt
from collections import defaultdict

from datasets import Dataset
from unsloth import FastLanguageModel, PatchFastRL
from trl import GRPOTrainer, GRPOConfig

PatchFastRL('GRPO', FastLanguageModel)

ALLOWED_ACTIONS = [
    'query_regulations', 'analyze_image', 'check_advertiser_history',
    'request_landing_page', 'request_id_verification',
    'submit_audit', 'approve', 'reject',
]

class EnvClient:
    def __init__(self, url):
        self.url = url
    def reset(self, task_id):
        return requests.post(f'{self.url}/reset', json={'task_id': task_id}, timeout=8).json()
    def step(self, action):
        return requests.post(f'{self.url}/step', json={'action': action}, timeout=8).json()

def safe_step(client, action):
    for _ in range(3):
        try:
            return client.step(action)
        except:
            time.sleep(0.5)
    return {'reward': -0.3}

def extract_json(text):
    try:
        if '```' in text:
            text = text.split('```')[1]
            if text.startswith('json'):
                text = text[4:]
        return json.loads(text.strip())
    except:
        return None

print('✅ Helpers loaded')

## Cell 6 — Dataset

In [ ]:
BASE_SCENARIOS = [
    {'task_id': 'task_1_healthcare',
     'text': "Healthcare ad: 'miracle supplement cures disease'. No actions taken yet.",
     'actions_already_taken': [], 'setup_actions': []},
    {'task_id': 'task_2_financial',
     'text': "Financial ad: 'guaranteed 500% returns, zero risk'. No actions taken yet.",
     'actions_already_taken': [], 'setup_actions': []},
    {'task_id': 'task_3_multimodal',
     'text': 'Multimodal ad: image may contain hidden violation. No actions taken yet.',
     'actions_already_taken': [], 'setup_actions': []},
    {'task_id': 'task_6_conflict',
     'text': 'High-trust advertiser but policy borderline. No actions taken yet.',
     'actions_already_taken': [], 'setup_actions': []},
    {'task_id': 'task_7_ambiguous',
     'text': 'Ambiguous wellness ad. Policy confidence low. No actions taken yet.',
     'actions_already_taken': [], 'setup_actions': []},
    {'task_id': 'task_8_adversarial',
     'text': 'Natural supplement ad. Image may hide violation. No actions taken yet.',
     'actions_already_taken': [], 'setup_actions': []},
    {'task_id': 'task_9_dependency_trap',
     'text': 'Certified wellness product. Regulations and CRM look clean. No actions taken yet.',
     'actions_already_taken': [], 'setup_actions': []},
    {'task_id': 'task_10_failure',
     'text': 'Healthcare ad. First API call may fail. No actions taken yet.',
     'actions_already_taken': [], 'setup_actions': []},
    # task_4 targeting — fresh
    {'task_id': 'task_4_targeting',
     'text': "Financial ad targeting young users: 'Start Your First Investment Portfolio'. No actions taken yet.",
     'actions_already_taken': [], 'setup_actions': []},
    # task_4 targeting — mid state
    {'task_id': 'task_4_targeting',
     'text': 'Financial ad targeting young users. Policy queried, need to verify age targeting.',
     'actions_already_taken': ['query_regulations'],
     'setup_actions': [{'action_type': 'query_regulations', 'reasoning': 'policy lookup'}]},
    # task_4 targeting — audit ready
    {'task_id': 'task_4_targeting',
     'text': 'Financial ad targeting minors. Policy, history, and ID verification done. Submit audit.',
     'actions_already_taken': ['query_regulations', 'check_advertiser_history', 'request_id_verification'],
     'setup_actions': [
         {'action_type': 'query_regulations', 'reasoning': 'policy lookup'},
         {'action_type': 'check_advertiser_history', 'reasoning': 'trust score'},
         {'action_type': 'request_id_verification', 'reasoning': 'age check'},
     ]},
    {'task_id': 'task_1_healthcare',
     'text': 'Healthcare ad. Policy already queried.',
     'actions_already_taken': ['query_regulations'],
     'setup_actions': [{'action_type': 'query_regulations', 'reasoning': 'policy lookup'}]},
    {'task_id': 'task_2_financial',
     'text': 'Financial ad. Policy and history checked. Submit audit next.',
     'actions_already_taken': ['query_regulations', 'check_advertiser_history'],
     'setup_actions': [
         {'action_type': 'query_regulations', 'reasoning': 'policy lookup'},
         {'action_type': 'check_advertiser_history', 'reasoning': 'trust score'},
     ]},
    {'task_id': 'task_2_financial',
     'text': 'Financial ad. Policy, history, audit all complete. Make final decision.',
     'actions_already_taken': ['query_regulations', 'check_advertiser_history', 'submit_audit'],
     'setup_actions': [
         {'action_type': 'query_regulations', 'reasoning': 'policy lookup'},
         {'action_type': 'check_advertiser_history', 'reasoning': 'trust score'},
         {'action_type': 'submit_audit', 'reasoning': 'audit log'},
     ]},
]

PROMPT_TEMPLATE = """You are an enterprise Ad Policy Compliance Agent.

You MUST choose exactly ONE action_type from this list (any other value is invalid):
- query_regulations
- analyze_image
- check_advertiser_history
- request_landing_page
- request_id_verification
- submit_audit
- approve
- reject

REQUIRED PHASE ORDER:
1. query_regulations  -> always first
2. analyze_image / check_advertiser_history  -> gather signals
3. submit_audit  -> always before final decision
4. approve OR reject  -> only after audit

HARD RULES:
- NEVER repeat an action listed in `actions_already_taken`.
- Respond with ONLY a valid JSON object. No markdown, no prose.

Required format:
{{\"action_type\": \"<one_of_the_actions_above>\", \"reasoning\": \"<short reason>\"}}

Scenario: {text}
actions_already_taken: {actions_already_taken}

Your next action?"""

def build_dataset():
    rows = []
    for s in BASE_SCENARIOS:
        prompt = PROMPT_TEMPLATE.format(
            text=s['text'],
            actions_already_taken=json.dumps(s['actions_already_taken']),
        )
        rows.append({'prompt': prompt, 'task_id': s['task_id'], 'setup_actions': s['setup_actions']})
    return Dataset.from_list(rows * 10)

dataset = build_dataset()
print(f'✅ Dataset: {len(dataset)} examples')

## Cell 7 — Reward Function

In [ ]:
# Track rewards for plotting
reward_log = []
step_log = []
global_step_counter = [0]

def reward_environment(prompts, completions, task_id=None, setup_actions=None, **kwargs):
    client = EnvClient(ENV_URL)
    rewards = []

    for completion, t_id, setup in zip(completions, task_id, setup_actions):
        parsed = extract_json(completion)
        if not parsed:
            rewards.append(-1.0)
            continue

        action_type = parsed.get('action_type')
        if action_type not in ALLOWED_ACTIONS:
            rewards.append(-1.0)
            continue

        action = {
            'action_type': action_type,
            'reasoning': parsed.get('reasoning', 'format-compliant'),
        }

        try:
            client.reset(t_id)
            for s in setup:
                safe_step(client, s)

            result = safe_step(client, action)
            env_reward = float(result.get('reward', -0.2))
            status_msg = (result.get('status_message') or '').lower()

            rejected = (
                'api failure' in status_msg
                or 'invalid action' in status_msg
                or 'must call' in status_msg
            )
            shaped = -0.5 if rejected else 0.5 + env_reward
            rewards.append(shaped)

        except Exception:
            rewards.append(-0.3)

    # Log for plot
    avg = sum(rewards) / len(rewards) if rewards else 0.0
    global_step_counter[0] += 1
    reward_log.append(avg)
    step_log.append(global_step_counter[0])

    return rewards

print('✅ Reward function ready')

## Cell 8 — Load Model

In [ ]:
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    _vram = _props.total_memory
    _cc = (_props.major, _props.minor)
    print(f'GPU: {_props.name}  VRAM: {_vram / 1024**3:.1f} GB  Compute: {_cc[0]}.{_cc[1]}')
else:
    _vram, _cc = 0, (0, 0)

USE_4BIT = _vram < 40 * 1024**3   # T4/L4 → 4-bit; A100 → full precision
USE_BF16 = _cc >= (8, 0) and not USE_4BIT  # bf16 only with full-precision weights; 4-bit LoRA uses fp16
print(f'4-bit: {USE_4BIT}  bf16: {USE_BF16}')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.1-8B-Instruct',
    load_in_4bit=USE_4BIT,
    max_seq_length=2048,
    dtype=torch.float16 if USE_4BIT else None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=64,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('✅ Model loaded')

## Cell 9 — Train

In [ ]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_environment],
    args=GRPOConfig(
        output_dir='outputs',
        learning_rate=2e-5,
        num_train_epochs=3,
        per_device_train_batch_size=2 if not USE_4BIT else 1,
        gradient_accumulation_steps=4,
        num_generations=4 if not USE_4BIT else 2,
        max_prompt_length=768,
        max_completion_length=128,
        logging_steps=5,
        warmup_steps=10,
        bf16=USE_BF16,
        fp16=not USE_BF16,
        report_to='none',
    ),
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('Starting GRPO training...')
print(f'  bf16={USE_BF16}  fp16={not USE_BF16}  batch={2 if not USE_4BIT else 1}  gens={4 if not USE_4BIT else 2}')
trainer.train()
print('Training complete')

## Cell 10 — Plot Reward Curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

os.makedirs('outputs', exist_ok=True)

def moving_avg(data, window=5):
    if len(data) < window:
        return data
    return list(np.convolve(data, np.ones(window)/window, mode='valid'))

hist = pd.DataFrame(trainer.state.log_history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Reward curve (from our custom log) ---
ax = axes[0]
ax.plot(step_log, reward_log, alpha=0.3, color='steelblue', label='Raw')
smoothed = moving_avg(reward_log)
ax.plot(range(len(smoothed)), smoothed, color='steelblue', linewidth=2, label='Smoothed (MA-5)')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Reward Eval Step')
ax.set_ylabel('Avg Reward per Batch')
ax.set_title('Reward Curve')
ax.legend()
ax.grid(alpha=0.3)

# --- Plot 2: Loss curve (from trainer logs) ---
ax = axes[1]
loss_rows = hist.dropna(subset=['loss']) if 'loss' in hist.columns else pd.DataFrame()
if not loss_rows.empty:
    ax.plot(loss_rows['step'], loss_rows['loss'], color='#7c3aed', linewidth=2)
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Loss')
    ax.set_title('GRPO Loss')
    ax.grid(alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No loss data logged', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('GRPO Loss')

# --- Plot 3: Reward from trainer logs (if available) ---
ax = axes[2]
reward_cols = [c for c in hist.columns if 'reward' in c.lower() and 'std' not in c.lower()]
if reward_cols:
    col = reward_cols[0]
    rr = hist.dropna(subset=[col])
    ax.plot(rr['step'], rr[col], color='#16a34a', linewidth=2)
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Training Step')
    ax.set_ylabel(col)
    ax.set_title('Trainer Reward Log')
    ax.grid(alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No trainer reward data', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Trainer Reward Log')

plt.tight_layout()
plt.savefig('outputs/training_plots.png', dpi=150)
plt.show()
print('Saved to outputs/training_plots.png')

n = len(reward_log)
first_10 = reward_log[:min(10, n)]
last_10  = reward_log[max(0, n-10):]
print(f'\n--- Before vs After ---')
print(f'Avg reward (first 10 steps): {sum(first_10)/len(first_10):.3f}')
print(f'Avg reward (last 10 steps) : {sum(last_10)/len(last_10):.3f}')

## Cell 11 — Before vs After: Baseline Comparison

In [ ]:
from unsloth import FastLanguageModel as FLM

FLM.for_inference(model)

test_scenarios = [
    ('task_1_healthcare', "Healthcare ad: 'miracle cure'. No actions taken yet.", []),
    ('task_2_financial', "Financial ad: 'guaranteed returns'. No actions taken yet.", []),
    ('task_4_targeting', "Financial ad targeting teens. No actions taken yet.", []),
    ('task_2_financial', "Financial ad. Policy, history, audit done. Decide.",
     ['query_regulations', 'check_advertiser_history', 'submit_audit']),
]

print('=== Trained Model Outputs ===\n')
for task, text, taken in test_scenarios:
    prompt = PROMPT_TEMPLATE.format(text=text, actions_already_taken=json.dumps(taken))
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=64, temperature=0.1, do_sample=True)
    decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    parsed = extract_json(decoded) or decoded.strip()[:120]
    print(f'[{task}] taken={taken}')
    print(f'  -> {parsed}\n')

## Cell 12 — Save + Push to HF Hub

In [ ]:
model.save_pretrained('outputs/lora_adapter')
tokenizer.save_pretrained('outputs/lora_adapter')
print('LoRA adapter saved')

print('Merging adapter into base model...')
merged_model, merged_tokenizer = FastLanguageModel.from_pretrained(
    model_name='outputs/lora_adapter',
    load_in_4bit=False,
    max_seq_length=2048,
)
merged_model.save_pretrained_merged(
    'outputs/merged',
    merged_tokenizer,
    save_method='merged_16bit',
)
print('Merged model saved to outputs/merged')

if HF_REPO and HF_TOKEN:
    print(f'Pushing to {HF_REPO}...')
    merged_model.push_to_hub_merged(
        HF_REPO,
        merged_tokenizer,
        save_method='merged_16bit',
        token=HF_TOKEN,
    )
    print(f'Model live at https://huggingface.co/{HF_REPO}')
else:
    print('Set HF_REPO and HF_TOKEN in Cell 3 to push to Hub')

print('Done.')